In [ ]:
# Cell 1: Import libraries

from pathlib import Path
from glob import glob
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from monai.data import CacheDataset, DataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, Lambdad,
    RandFlipd, RandRotate90d, EnsureTyped,
    Activationsd, AsDiscreted
)
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.engines import SupervisedTrainer, SupervisedEvaluator
from monai.handlers import (
    StatsHandler, ValidationHandler, CheckpointSaver,
    TensorBoardStatsHandler, TensorBoardImageHandler,
    MeanDice, from_engine
)
from monai.inferers import SimpleInferer
from monai.utils import set_determinism

# Set seed
set_determinism(42)

root = Path("segmentation_data")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 8
max_epochs = 50
log_dir = "seg_binary"
ckpt_dir = "ckpt"

def files(split):
    imgs = {Path(p).stem: p for p in glob(str(root / f"{split}-images" / "*.png"))}
    lbls = {Path(p).stem: p for p in glob(str(root / f"{split}-labels" / "*.png"))}
    keys = sorted(imgs.keys() & lbls.keys())
    if not keys:
        raise RuntimeError(f"No matched image/label pairs found for split {split}.")
    return [{"image": imgs[k], "label": lbls[k]} for k in keys]

# Augmentations during training
train_tf = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys="image"),
    Lambdad(keys="label", func=lambda x: (x > 0).astype(np.float32)),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    EnsureTyped(keys=["image", "label"]),
])

# Augmentations during evaluation
eval_tf = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys="image"),
    Lambdad(keys="label", func=lambda x: (x > 0).astype(np.float32)),
    EnsureTyped(keys=["image", "label"]),
])

post = Compose([
    Activationsd(keys="pred", sigmoid=True),
    AsDiscreted(keys=["pred", "label"], threshold=0.5),
])

# Data loaders
train_loader = DataLoader(
    CacheDataset(files("train"), train_tf, cache_rate=1.0),
    batch_size=batch_size, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    CacheDataset(files("valid"), eval_tf, cache_rate=1.0),
    batch_size=batch_size, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    CacheDataset(files("test"), eval_tf, cache_rate=1.0),
    batch_size=batch_size, shuffle=False, num_workers=0
)

# Model definitions
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# Loss functions and optimizers
loss_function = DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
inferer = SimpleInferer()
writer = SummaryWriter(log_dir=log_dir)

!rm -f seg_binary/*

In [ ]:
# Cell 2: Trainer and Evaluator

# evaluator (includes logic for posting on Tensorboard)
val_evaluator = SupervisedEvaluator(
    device=device,
    val_data_loader=val_loader,
    network=model,
    inferer=inferer,
    postprocessing=post,
    key_val_metric={
        "val_mean_dice": MeanDice(
            include_background=True,
            output_transform=from_engine(["pred", "label"]),
        )
    },
    val_handlers=[
        StatsHandler(
            output_transform=lambda x: None,
            iteration_log=False,
        ),
        TensorBoardStatsHandler(
            summary_writer=writer,
            tag_name="val",
            output_transform=lambda x: None,
            iteration_log=False,
        ),
        TensorBoardImageHandler(
            summary_writer=writer,
            interval=1,
            epoch_level=True,
            batch_transform=from_engine(["image", "label"]),
            output_transform=from_engine(["pred"]),
            index=0,
            max_channels=1,
        ),
        CheckpointSaver(
            save_dir=ckpt_dir,
            save_dict={"model": model},
            save_key_metric=True,
            key_metric_name="val_mean_dice",
            key_metric_filename="best_model.pt",
        ),
    ],
)

# Trainer 
trainer = SupervisedTrainer(
    device=device,
    max_epochs=max_epochs,
    train_data_loader=train_loader,
    network=model,
    optimizer=optimizer,
    loss_function=loss_function,
    inferer=inferer,
    postprocessing=post,
    train_handlers=[
        ValidationHandler(validator=val_evaluator, interval=1, epoch_level=True),
        TensorBoardStatsHandler(
            summary_writer=writer,
            tag_name="train_loss",
            output_transform=from_engine(["loss"], first=True),
        ),
    ],
)


In [ ]:
# Cell 3: Load Tensorboard
import os
os.environ["TENSORBOARD_BINARY"] = "/home/jupyter/miniconda3/bin/tensorboard"
%load_ext tensorboard
%tensorboard --host 0.0.0.0  --logdir seg_binary --reload_interval 30

In [ ]:
# Cell 4: Run the trainer! 
trainer.run()

In [ ]:
# Cell 5: Load the best checkpoint and run the test set

ckpt = torch.load(f"{ckpt_dir}/best_model.pt", map_location=device)
model.load_state_dict(ckpt)

test_evaluator = SupervisedEvaluator(
    device=device,
    val_data_loader=test_loader,
    network=model,
    inferer=inferer,
    postprocessing=post,
    key_val_metric={
        "test_mean_dice": MeanDice(
            include_background=True,
            output_transform=from_engine(["pred", "label"]),
        )
    },
    val_handlers=[
        StatsHandler(output_transform=lambda x: None),
    ],
)

test_evaluator.run()
print("Test Dice:", test_evaluator.state.metrics["test_mean_dice"])

In [ ]:
# Cell 6: Review a few cases

%matplotlib widget
import matplotlib.pyplot as plt
import torch

model.eval()

with torch.no_grad():
    batch = next(iter(test_loader))
    images = batch["image"].to(device)
    labels = batch["label"].to(device)

    preds = inferer(images, model)
    preds = (preds > 0.5).float()

# pick which batch indices to show
example_idxs = torch.randperm(images.shape[0])[:3].tolist()

fig, axes = plt.subplots(len(example_idxs), 3, figsize=(12, 8), constrained_layout=True)

for row, idx in enumerate(example_idxs):
    img = images[idx].cpu().squeeze().numpy()
    lbl = labels[idx].cpu().squeeze().numpy()
    pred = preds[idx].cpu().squeeze().numpy()

    axes[row, 0].imshow(img, cmap="gray", aspect="auto")
    axes[row, 0].set_title("Test Image" if row == 0 else "", pad=10)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(lbl, cmap="gray", aspect="auto")
    axes[row, 1].set_title("Ground Truth" if row == 0 else "", pad=10)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(pred, cmap="gray", aspect="auto")
    axes[row, 2].set_title("Prediction" if row == 0 else "", pad=10)
    axes[row, 2].axis("off")

plt.show()